# FNN to TVB Soft Visual Replacement

This notebook implements a first soft-replacement prototype for visual nodes.

Compared with the one-way coupling notebook, the goal here is different:
- one-way coupling: FNN output is injected as a regional stimulus
- soft replacement: TVB visual nodes are still part of the whole-brain scaffold, but their local state is blended toward an FNN-derived target at every integration step

This is still a prototype, not a final node-replacement framework. The current implementation only modifies the first state variable of the selected visual nodes.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "fnn").exists() and (candidate / "notebook").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from fnn_tvb_notebook_utils import ensure_src_on_sys_path
ensure_src_on_sys_path(PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from fnn_tvb_notebook_utils import (
    build_constant_covariates,
    build_random_fnn,
    build_sampled_region_stimulus,
    build_tvb_simulator,
    extract_mean_region_trace,
    extract_region_trace,
    frame_times_ms,
    load_pretrained_fnn,
    make_moving_bar_video,
    map_drive_to_reference_quantiles,
    predict_responses,
    run_tvb,
    run_tvb_with_soft_replacement,
    set_reproducible_seed,
    summarize_population_drive,
    unpack_first_monitor,
)

set_reproducible_seed(7)

OUTPUT_DIR = PROJECT_ROOT / "notebook" / "output" / "fnn_tvb_soft_visual_replacement"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSION = 8
SCAN_IDX = 5
PRETRAINED_DIR = PROJECT_ROOT / "notebook" / "data" / "fnn" / "microns_digital_twin" / "params"
PRETRAINED_READY = (PRETRAINED_DIR / "params_core.pt").exists() and (PRETRAINED_DIR / f"params_{SESSION}_{SCAN_IDX}.pt").exists()
USE_PRETRAINED = PRETRAINED_READY

FRAME_COUNT = 90
FPS = 30.0
TARGET_LABELS = ("rV1", "rV2", "lV1", "lV2")
CONTROL_LABEL = "rFEF"

STIMULUS_DRIVE_SCALE = 0.25
SOFT_BLEND_ALPHA = 0.35
SOFT_RANGE_QUANTILES = (0.1, 0.9)

print("Project root:", PROJECT_ROOT)
print("Output dir:", OUTPUT_DIR)
print("Pretrained ready:", PRETRAINED_READY)


In [ ]:
frames = make_moving_bar_video(frame_count=FRAME_COUNT)
perspectives, modulations = build_constant_covariates(len(frames))

if USE_PRETRAINED:
    model, unit_ids = load_pretrained_fnn(
        params_dir=PRETRAINED_DIR,
        session=SESSION,
        scan_idx=SCAN_IDX,
        cuda=False,
    )
    model_mode = f"pretrained MICrONS digital twin ({SESSION}-{SCAN_IDX})"
else:
    model = build_random_fnn(units=64, randomize=True)
    unit_ids = None
    model_mode = "randomized FNN skeleton"

responses = predict_responses(model, frames, perspectives, modulations)
drive = summarize_population_drive(responses, method="mean", normalize=True)
drive_times_ms = frame_times_ms(len(frames), fps=FPS)
simulation_length_ms = float(drive_times_ms[-1] + 100.0)

print("Model mode:", model_mode)
print("responses shape:", responses.shape)
print("drive min/max:", float(drive.min()), float(drive.max()))
print("simulation_length_ms:", simulation_length_ms)


In [ ]:
baseline = build_tvb_simulator(simulation_length_ms=simulation_length_ms)
baseline_results = run_tvb(baseline, simulation_length_ms)
baseline_times, baseline_values = unpack_first_monitor(baseline_results)
_, baseline_visual_mean = extract_mean_region_trace(
    baseline_times,
    baseline_values,
    baseline,
    TARGET_LABELS,
)

stimulus = build_sampled_region_stimulus(
    connectivity=baseline.connectivity,
    sample_times_ms=drive_times_ms,
    sample_values=STIMULUS_DRIVE_SCALE * drive,
    target_labels=TARGET_LABELS,
    amplitude=1.0,
)
stimulated = build_tvb_simulator(simulation_length_ms=simulation_length_ms, stimulus=stimulus)
stimulated_results = run_tvb(stimulated, simulation_length_ms)
stimulated_times, stimulated_values = unpack_first_monitor(stimulated_results)

soft_target_series = map_drive_to_reference_quantiles(
    drive,
    baseline_visual_mean,
    lower_quantile=SOFT_RANGE_QUANTILES[0],
    upper_quantile=SOFT_RANGE_QUANTILES[1],
)
soft_simulator = build_tvb_simulator(simulation_length_ms=simulation_length_ms)
soft_results, replacement_times_ms, replacement_values = run_tvb_with_soft_replacement(
    soft_simulator,
    simulation_length_ms=simulation_length_ms,
    sample_times_ms=drive_times_ms,
    sample_values=soft_target_series,
    target_labels=TARGET_LABELS,
    blend_alpha=SOFT_BLEND_ALPHA,
    variable_index=0,
)
soft_times, soft_values = unpack_first_monitor(soft_results)


In [ ]:
_, baseline_visual_mean = extract_mean_region_trace(baseline_times, baseline_values, baseline, TARGET_LABELS)
_, stimulated_visual_mean = extract_mean_region_trace(stimulated_times, stimulated_values, stimulated, TARGET_LABELS)
_, soft_visual_mean = extract_mean_region_trace(soft_times, soft_values, soft_simulator, TARGET_LABELS)

_, baseline_control = extract_region_trace(baseline_times, baseline_values, baseline, CONTROL_LABEL)
_, stimulated_control = extract_region_trace(stimulated_times, stimulated_values, stimulated, CONTROL_LABEL)
_, soft_control = extract_region_trace(soft_times, soft_values, soft_simulator, CONTROL_LABEL)


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=False)

axes[0].plot(drive_times_ms, drive, color="tab:blue", lw=2, label="Normalized FNN drive")
axes[0].plot(drive_times_ms, soft_target_series, color="tab:orange", lw=2, label="Soft replacement target")
axes[0].set_title("FNN drive and soft replacement target")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

axes[1].plot(baseline_times, baseline_visual_mean, ls="--", color="black", label="Baseline")
axes[1].plot(stimulated_times, stimulated_visual_mean, color="tab:green", label="One-way stimulus")
axes[1].plot(soft_times, soft_visual_mean, color="tab:red", label="Soft replacement")
axes[1].set_title("Mean visual-node response")
axes[1].set_ylabel("State variable V")
axes[1].legend()

for label in TARGET_LABELS:
    _, baseline_trace = extract_region_trace(baseline_times, baseline_values, baseline, label)
    _, soft_trace = extract_region_trace(soft_times, soft_values, soft_simulator, label)
    axes[2].plot(baseline_times, baseline_trace, ls="--", alpha=0.6, label=f"{label} baseline")
    axes[2].plot(soft_times, soft_trace, lw=1.5, label=f"{label} soft")
axes[2].set_title("Per-node visual traces: baseline vs soft replacement")
axes[2].set_ylabel("State variable V")
axes[2].legend(ncol=2, fontsize=8)

axes[3].plot(baseline_times, baseline_control, ls="--", color="black", label=f"{CONTROL_LABEL} baseline")
axes[3].plot(stimulated_times, stimulated_control, color="tab:green", label=f"{CONTROL_LABEL} one-way")
axes[3].plot(soft_times, soft_control, color="tab:red", label=f"{CONTROL_LABEL} soft")
axes[3].set_title("Control-region response")
axes[3].set_xlabel("Time (ms)")
axes[3].set_ylabel("State variable V")
axes[3].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "fnn_tvb_soft_visual_replacement.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
summary = {
    "model_mode": model_mode,
    "soft_blend_alpha": SOFT_BLEND_ALPHA,
    "soft_range_quantiles": list(SOFT_RANGE_QUANTILES),
    "visual_mean_delta_one_way": float(np.mean(stimulated_visual_mean - baseline_visual_mean)),
    "visual_mean_delta_soft": float(np.mean(soft_visual_mean - baseline_visual_mean)),
    "control_mean_delta_one_way": float(np.mean(stimulated_control - baseline_control)),
    "control_mean_delta_soft": float(np.mean(soft_control - baseline_control)),
    "replacement_value_min": float(np.min(replacement_values)),
    "replacement_value_max": float(np.max(replacement_values)),
}
summary


In [ ]:
import json

summary_path = OUTPUT_DIR / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
np.savez(
    OUTPUT_DIR / "fnn_tvb_soft_visual_replacement_data.npz",
    drive_times_ms=drive_times_ms,
    drive=drive,
    soft_target_series=soft_target_series,
    replacement_times_ms=replacement_times_ms,
    replacement_values=replacement_values,
    baseline_times=baseline_times,
    baseline_values=baseline_values,
    stimulated_times=stimulated_times,
    stimulated_values=stimulated_values,
    soft_times=soft_times,
    soft_values=soft_values,
)

print("Saved summary to:", summary_path)
print("Saved arrays to:", OUTPUT_DIR / "fnn_tvb_soft_visual_replacement_data.npz")


## Design Notes

The current soft-replacement rule is:

1. run a baseline TVB simulation
2. estimate the baseline dynamic range of the visual nodes
3. map the FNN drive into that range
4. during a second TVB run, blend the selected visual-node state toward the mapped FNN target at every integration step

In code, the update is approximately:

`x_visual <- (1 - alpha) * x_visual + alpha * x_fnn_target`

This keeps the visual nodes inside the TVB graph, but partially shifts their local state toward an FNN-derived representation.
